# Tren topik/sentimen per waktu · review
Volume asli, proporsi topik, sentimen model dan pergeseran distribusi. Empat pendekatan topik: NMF, LDA, TF-IDF KMeans, E5 KMeans. Tiga pendekatan sentimen: leksikon negasi-aware, classifier pretrained, NLI zero-shot.

Model topik fit periode awal, dipilih pada periode tengah, dan diaudit pada periode akhir. Seluruh label topik berasal dari model beku yang sama agar ID topik dapat dibandingkan. Sentimen tanpa gold hanyalah estimasi, agreement antar model bukan accuracy. Perubahan waktu tidak membuktikan sebab-akibat.

## 00 · Konfigurasi

Ubah sumber/kolom/ukuran sampel di sini. Tidak ada grafik otomatis.

In [1]:
SEED,TOP=42,15
RESULT_NAME='hasil_temporal_review'
TEXT_COL,TIME_COL='comment','published_at'
N_ROWS,N_FIT,EDA_N,N_SENTIMENT=None,5000,3000,1500
FREQ,TIMEZONE,MIN_PERIOD='W','Asia/Jakarta',30
K_GRID=[4,6,8]
DOMAIN_STOP=set()

## 01 · Lingkungan & preprocessing teknis

Safe cleaning: encoding, NFC, HTML entity, invisible, whitespace. Pertahankan angka, negasi, case, entitas dan bahasa. Tanpa stemming/stopword removal pada encoder.

In [2]:
import os,sys,subprocess,time,json,re,html,unicodedata,random,gc
from pathlib import Path
from collections import Counter
assert Path('/kaggle/input').exists(),'Jalankan di Kaggle.'
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',HF_HUB_DISABLE_PROGRESS_BARS='1')
subprocess.run([sys.executable,'-m','pip','install','-q','ftfy>=6.3,<7','rank-bm25>=0.2,<1','sentencepiece>=0.2,<1','langid==1.1.6'],check=True)
import numpy as np,pandas as pd,torch,ftfy
from IPython.display import display
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Ridge,LogisticRegression
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
assert torch.cuda.is_available(),'Aktifkan GPU Kaggle.'
OUT=Path('/kaggle/working')/RESULT_NAME;OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns',None,'display.max_colwidth',120)
START=time.perf_counter()

In [3]:
def show(title,table):
    print('\n'+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(4))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({'n':n,'%':100*n/(len(values) or np.nan)})

def clean(value):
    if not isinstance(value,str): return ''
    repaired=ftfy.fix_text(value,unescape_html=False,uncurl_quotes=False,fix_latin_ligatures=False,
        fix_character_width=False,remove_control_chars=False,normalization=None)
    text=unicodedata.normalize('NFC',html.unescape(repaired))
    text=text.translate(dict.fromkeys(map(ord,'\ufeff\u200b\u2060\x00'),None))
    text=re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

def key(value): return clean(value).casefold()
def cap(frame,n): return frame.sample(min(n,len(frame)),random_state=SEED).reset_index(drop=True) if n else frame.reset_index(drop=True)
def locate(folder,repo):
    matches=list(Path('/kaggle/input').glob('**/models/'+folder+'/config.json'))
    return str(matches[0].parent) if matches else repo
NOISE={'url':r'https?://|www\.', 'mention':r'@\w+','hashtag':r'#\w+','angka':r'\d',
 'emoji/simbol':r'[\U0001F300-\U0001FAFF\u2600-\u27BF]','huruf_ulang':r'([^\W\d_])\1{2,}',
 'baca_ulang':r'[!?.,]{2,}','html':r'</?[a-zA-Z][^>]*>','encoding':r'\ufffd|Ã.|Â.',
 'non_ascii':r'[^\x00-\x7f]','spasi_ulang':r' {2,}','invisible':r'[\u200b\ufeff]',
 'KAPITAL':r'(?-i:\b[A-Z]{3,}\b)','huruf_angka':r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
 'slang_contoh':r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
 'negasi':r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'}

In [4]:
def text_eda(values,title):
    valid=values.where(values.map(lambda v:isinstance(v,str)),'').fillna('')
    show(title+' kualitas',pd.Series({'n':len(values),'null':values.isna().sum(),'kosong/spasi':valid.str.strip().eq('').sum(),
        'unik_terisi':valid[valid.str.strip().ne('')].nunique(),'duplikat_ekstra_terisi':valid[valid.str.strip().ne('')].duplicated().sum()}))
    show(title+' length',pd.DataFrame({'karakter':valid.str.len(),'kata':valid.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T)
    sample=valid[valid.str.strip().ne('')].sample(min(EDA_N,valid.str.strip().ne('').sum()),random_state=SEED)
    tokens=sample.str.casefold().str.findall(r'[^\W\d_]+')
    count=pd.Series(Counter(w for seq in tokens for w in seq),dtype='int64').sort_values(ascending=False)
    show(title+' vocabulary sampel',pd.Series({'n_sample':len(sample),'token':count.sum(),'unik':len(count),
        'hapax':count.eq(1).sum(),'hapax_%_tipe':100*count.eq(1).mean(),'hapax_%_token':100*count.eq(1).sum()/(count.sum() or np.nan)}))
    show(title+' top words',count.head(TOP).rename('n').to_frame().assign(pct=count.head(TOP)/(count.sum() or np.nan)*100))
    for size in [2,3]:
        grams=pd.Series(Counter(' '.join(seq[i:i+size]) for seq in tokens for i in range(len(seq)-size+1)),dtype='int64').sort_values(ascending=False)
        show(title+f' top {size}-gram',grams.head(TOP))
    flags=pd.DataFrame({name:valid.str.count(regex,flags=re.I).gt(0) for name,regex in NOISE.items()})
    show(title+' noise · % semua baris, dapat overlap',pd.DataFrame({'n':flags.sum(),'%':flags.mean()*100}))
    terms=count.index.to_series().astype('string')
    weird=terms.str.len().ge(20)|terms.str.count(r'([^\W\d_])\1{2,}').gt(0)
    show(title+' kandidat kata aneh, bukan pasti typo',count[weird].head(TOP))
    from langid.langid import LanguageIdentifier,model
    from threadpoolctl import threadpool_limits
    detector=LanguageIdentifier.from_modelstring(model,norm_probs=True)
    lang_sample=sample.sample(min(150,len(sample)),random_state=SEED)
    with threadpool_limits(limits=1):
        lang=lang_sample.map(lambda s:detector.classify(s[:3000]) if len(s.split())>=5 else ('tidak_pasti',np.nan))
    show(title+' estimasi bahasa · % sampel, bukan gold',distribution(lang.map(lambda p:p[0] if p[1]>=.8 else 'tidak_pasti')))

In [5]:
_encoder=None;_encoder_tok=None
@torch.inference_mode()
def embed(texts,prefix='query: '):
    global _encoder,_encoder_tok
    if _encoder is None:
        path=locate('e5','intfloat/multilingual-e5-base')
        _encoder_tok=AutoTokenizer.from_pretrained(path)
        _encoder=AutoModel.from_pretrained(path).cuda().eval()
    texts=list(texts);output=[]
    for start in range(0,len(texts),32):
        batch=_encoder_tok([prefix+t for t in texts[start:start+32]],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):hidden=_encoder(**batch).last_hidden_state
        mask=batch.attention_mask.unsqueeze(-1)
        pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1)
        output.append(torch.nn.functional.normalize(pooled,dim=1).cpu().numpy())
    return np.concatenate(output)

def release_encoder():
    global _encoder,_encoder_tok
    _encoder=None;_encoder_tok=None;gc.collect();torch.cuda.empty_cache()

In [6]:
@torch.inference_mode()
def pair_logits(repo,frame,folder=None,max_len=256):
    path=locate(folder,repo) if folder else repo
    tok=AutoTokenizer.from_pretrained(path);model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    mapping={str(k).lower():int(v) for k,v in model.config.label2id.items()};output=[]
    for start in range(0,len(frame),32):
        batch=frame.iloc[start:start+32]
        enc=tok(batch.a.tolist(),batch.b.tolist(),padding=True,truncation=True,max_length=max_len,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):logits=model(**enc).logits
        output.append(logits.float().cpu().numpy())
    del model;gc.collect();torch.cuda.empty_cache()
    return np.concatenate(output),mapping

## 02 · Waktu, volume dan kualitas

Missing diaudit sebelum drop. Duplikat dipertahankan untuk volume; fitting topik memakai unique text periode awal. Jika N_ROWS dibatasi, volume yang dilaporkan adalah volume sampel, bukan seluruh sumber.

In [7]:
path=next(Path('/kaggle/input').rglob('komentar-mbg.csv'));raw=pd.read_csv(path)
assert {TEXT_COL,TIME_COL}.issubset(raw),raw.columns.tolist()
show('Missing sumber',pd.DataFrame({'n':raw.isna().sum(),'%':raw.isna().mean()*100}))
raw['text']=raw[TEXT_COL].map(clean);raw['time']=pd.to_datetime(raw[TIME_COL],errors='coerce',format='mixed',utc=True)
print('Invalid date:',raw.time.isna().sum(),'| teks kosong:',raw.text.eq('').sum())
data=cap(raw.loc[raw.time.notna()&raw.text.ne('')].copy(),N_ROWS).sort_values('time').reset_index(drop=True)
data['period']=data.time.dt.tz_convert(TIMEZONE).dt.tz_localize(None).dt.to_period(FREQ).dt.start_time
periods=sorted(data.period.unique());assert len(periods)>=8,'Perlu >=8 periode; sesuaikan FREQ/data.'
cut1,cut2=periods[int(.6*len(periods))],periods[int(.8*len(periods))]
early=data.period<cut1;middle=(data.period>=cut1)&(data.period<cut2);late=data.period>=cut2
fit=cap(data.loc[early].drop_duplicates('text'),N_FIT)
assert min(early.sum(),middle.sum(),late.sum())>=30
show('Volume per periode',data.groupby('period').agg(n=('text','size'),unik=('text',lambda s:s.map(key).nunique())))
print('Waktu:',data.time.min(),data.time.max(),'| training sebelum:',cut1,'| final sejak:',cut2)


Missing sumber


,n,%
author,0,0.0
comment,0,0.0
likes,0,0.0
published_at,0,0.0


Invalid date: 0 | teks kosong: 0



Volume per periode


,n,unik
period,,
2026-02-16,3547,3405
2026-02-23,3853,2300
2026-03-02,169,92
2026-03-09,137,105
2026-03-16,150,113
2026-03-23,170,119
2026-03-30,121,78
2026-04-06,95,61
2026-04-13,193,122


Waktu: 2026-02-17 12:49:14+00:00 2026-06-24 10:54:03+00:00 | training sebelum: 2026-05-04 00:00:00 | final sejak: 2026-06-01 00:00:00


## 03 · EDA temporal wajib

Audit volume/unik, panjang/noise, distribusi hari/jam, kelengkapan periode, dan kosakata awal vs akhir. Tidak menghapus periode kecil diam-diam.

In [8]:
text_eda(data.text,'Teks seluruh rentang')
show('Hari lokal',distribution(data.time.dt.tz_convert(TIMEZONE).dt.dayofweek))
show('Jam lokal',distribution(data.time.dt.tz_convert(TIMEZONE).dt.hour))
volume=data.groupby('period').size().sort_index()
grid=pd.period_range(volume.index.min(),volume.index.max(),freq=FREQ).start_time
show('Periode + dukungan',volume.rename('n').to_frame().assign(cukup=volume>=MIN_PERIOD))
print('Missing bins (kalender):',len(set(grid)-set(volume.index)))
print('Duplikat ekstra seluruh rentang:',data.text.map(key).duplicated().sum())


Teks seluruh rentang kualitas


,0
n,15000
null,0
kosong/spasi,0
unik_terisi,11375
duplikat_ekstra_terisi,3625



Teks seluruh rentang length


,count,mean,std,min,1%,50%,90%,95%,99%,max
karakter,15000.0,112.6052,153.2294,1.0,5.0,78.0,230.0,319.0,601.06,8479.0
kata,15000.0,17.2615,23.1067,1.0,1.0,12.0,35.0,48.0,91.00,1306.0



Teks seluruh rentang vocabulary sampel


,0
n_sample,3000.0000
token,51029.0000
unik,8124.0000
hapax,4477.0000
hapax_%_tipe,55.1083
hapax_%_token,8.7734


... (16 outputs left out)


Periode + dukungan


,n,cukup
period,,
2026-02-16,3547,True
2026-02-23,3853,True
2026-03-02,169,True
2026-03-09,137,True
2026-03-16,150,True
2026-03-23,170,True
2026-03-30,121,True
2026-04-06,95,True
2026-04-13,193,True


Missing bins (kalender): 0


Duplikat ekstra seluruh rentang: 3644


## 04 · Representasi topic vs sentiment

Sentimen/embedding memakai safe text. BoW memakai normalisasi slang terkurasi + stopword fungsi, tanpa membuang negasi/angka. DOMAIN_STOP kosong secara default; tinjau kata domain sebelum menambahkan.

In [9]:
SLANG={'yg':'yang','utk':'untuk','gak':'tidak','ga':'tidak','nggak':'tidak','tdk':'tidak','blm':'belum','bgt':'banget','udh':'sudah','klo':'kalau'}
STOP=set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split())|DOMAIN_STOP
STOP-=set('tidak tak bukan belum jangan tanpa kurang'.split())
def bow(text):
    text=re.sub(r'https?://\S+|@\w+',' ',text.casefold())
    words=[SLANG.get(w,w) for w in re.findall(r'[^\W_]+',text)]
    return ' '.join(w for w in words if w not in STOP and len(w)>1)
data['bow']=data.text.map(bow);fit['bow']=fit.text.map(bow)
cv=CountVectorizer(min_df=3,max_df=.95,max_features=20000)
counts_fit=cv.fit_transform(fit.bow);counts_all=cv.transform(data.bow)
from sklearn.feature_extraction.text import TfidfTransformer
tfidf=TfidfTransformer(sublinear_tf=True).fit(counts_fit)
x_fit=tfidf.transform(counts_fit);x_all=tfidf.transform(counts_all);vocab=cv.get_feature_names_out()
show('Before/after',data[['text','bow']].head(5))
print('OOV/empty BoW per waktu:',pd.Series(np.asarray(counts_all.sum(1)).ravel()==0).groupby(data.period).mean().to_dict())


Before/after


,text,bow
0,Dulu program diragukan akan berjalan .tetapi begitu berjalan banyak yg ketakutan kwatir tidak memperoleh suara pemil...,dulu program diragukan akan berjalan tetapi begitu berjalan banyak ketakutan kwatir tidak memperoleh suara pemilu ak...
1,"Betul skali, MBG ladang korupsi nyata menu sampe ke anak"" tidak sesuai budget. 😡😡😡",betul skali mbg ladang korupsi nyata menu sampe anak tidak sesuai budget
2,Pemahaman saya MBG itu tidak gratis Karena dari kami bayar pajak Jadi kami yang membayarnya,pemahaman mbg tidak gratis karena bayar pajak jadi membayarnya
3,https://youtube.com/shorts/9EMK3flLRw8?si=6wcx82LG8kTz5ZYC,
4,"MANTAB...tetap semangat,berjuang dengan suara lantang.",mantab tetap semangat berjuang suara lantang


OOV/empty BoW per waktu: {Timestamp('2026-02-16 00:00:00'): 0.021144629264166902, Timestamp('2026-02-23 00:00:00'): 0.016610433428497273, Timestamp('2026-03-02 00:00:00'): 0.011834319526627219, Timestamp('2026-03-09 00:00:00'): 0.043795620437956206, Timestamp('2026-03-16 00:00:00'): 0.02, Timestamp('2026-03-23 00:00:00'): 0.01764705882352941, Timestamp('2026-03-30 00:00:00'): 0.01652892561983471, Timestamp('2026-04-06 00:00:00'): 0.031578947368421054, Timestamp('2026-04-13 00:00:00'): 0.031088082901554404, Timestamp('2026-04-20 00:00:00'): 0.012658227848101266, Timestamp('2026-04-27 00:00:00'): 0.11267605633802817, Timestamp('2026-05-04 00:00:00'): 0.018867924528301886, Timestamp('2026-05-11 00:00:00'): 0.025423728813559324, Timestamp('2026-05-18 00:00:00'): 0.03260869565217391, Timestamp('2026-05-25 00:00:00'): 0.028169014084507043, Timestamp('2026-06-01 00:00:00'): 0.028659160696008188, Timestamp('2026-06-08 00:00:00'): 0.02400698384984723, Timestamp('2026-06-15 00:00:00'): 0.0220763

## 05 · Empat pendekatan topik, pilih hanya validation

Coherence dihitung pada dokumen validation dan top words train dengan vocabulary yang sama. Hindari membuat matriks vocabulary²: co-occurence hanya untuk kata topik. Label kategori tidak diperlukan. TQ adalah proxy, bukan kebenaran topik.

In [10]:
from sklearn.decomposition import NMF,LatentDirichletAllocation
from sklearn.cluster import MiniBatchKMeans
def top_terms(labels):
    result={}
    for topic in sorted(set(labels)-{-1}):
        mass=np.asarray(counts_fit[labels==topic].sum(0)).ravel()
        result[int(topic)]=np.argsort(-mass)[:10]
    return result
def coherence(topics,presence):
    scores=[];n=presence.shape[0]
    for words in topics.values():
        x=(presence[:,words]>0).astype('int64');freq=np.asarray(x.sum(0)).ravel()/n;joint=(x.T@x).toarray()/n
        for i in range(len(words)):
            for j in range(i):
                p=joint[i,j]
                scores.append(-1. if p==0 else 1. if p>=1 else np.log(p/max(1e-12,freq[i]*freq[j]))/-np.log(p))
    return float(np.mean(scores)) if scores else np.nan
models={};candidate=[]
def evaluate(name,model,fit_labels,all_labels):
    terms=top_terms(fit_labels);flat=[int(w) for topic in terms.values() for w in topic]
    npmi=coherence(terms,counts_all[middle.to_numpy()]);diversity=len(set(flat))/max(1,len(flat))
    share=pd.Series(all_labels[middle.to_numpy()]).value_counts(normalize=True)
    candidate.append({'name':name,'n_topics_fit':len(terms),'NPMI_val':npmi,'diversity':diversity,'TQ':(npmi+1)/2*diversity,'max_share_val':share.max()})
    models[name]={'model':model,'labels':all_labels,'terms':terms}

In [11]:
for k in K_GRID:
    nmf=NMF(n_components=k,init='nndsvda',random_state=SEED,max_iter=300).fit(x_fit)
    pred=nmf.transform(x_all).argmax(1);pred[np.asarray(counts_all.sum(1)).ravel()==0]=-1
    evaluate(f'nmf_k{k}',nmf,nmf.transform(x_fit).argmax(1),pred)
    lda=LatentDirichletAllocation(n_components=k,learning_method='online',max_iter=8,random_state=SEED,n_jobs=2).fit(counts_fit)
    pred=lda.transform(counts_all).argmax(1);pred[np.asarray(counts_all.sum(1)).ravel()==0]=-1
    evaluate(f'lda_k{k}',lda,lda.transform(counts_fit).argmax(1),pred)
    km=MiniBatchKMeans(n_clusters=k,random_state=SEED,n_init=5,batch_size=512).fit(x_fit)
    pred=km.predict(x_all);pred[np.asarray(counts_all.sum(1)).ravel()==0]=-1
    evaluate(f'tfidf_kmeans_k{k}',km,km.labels_,pred)
fit_e=embed(fit.text);all_e=embed(data.text);release_encoder()
for k in K_GRID:
    km=MiniBatchKMeans(n_clusters=k,random_state=SEED,n_init=5,batch_size=512).fit(fit_e)
    evaluate(f'e5_kmeans_k{k}',km,km.labels_,km.predict(all_e))
board=pd.DataFrame(candidate);pool=board[board.max_share_val.le(.8)&board.TQ.notna()&board.n_topics_fit.ge(2)]
assert len(pool),'Semua topik dominan/invalid; tinjau preprocessing dan grid.'
chosen=pool.sort_values('TQ',ascending=False).iloc[0]['name']
show('Model topik validation',board.sort_values('TQ',ascending=False));print('Dipilih:',chosen)
(OUT/'selection.json').write_text(json.dumps({'topic_model':chosen,'selection_end':str(cut2)},indent=2))


Model topik validation


,name,n_topics_fit,NPMI_val,diversity,TQ,max_share_val
0,nmf_k4,4,0.0979,0.8000,0.4392,0.4167
3,nmf_k6,6,0.1075,0.7500,0.4153,0.3685
1,lda_k4,4,0.1054,0.7250,0.4007,0.3423
2,tfidf_kmeans_k4,4,0.0805,0.7000,0.3782,0.6413
6,nmf_k8,8,0.0990,0.6375,0.3503,0.3031
7,lda_k8,8,0.0421,0.6250,0.3256,0.2500
4,lda_k6,6,0.0733,0.5833,0.3130,0.2312
9,e5_kmeans_k4,4,0.0224,0.4750,0.2428,0.3039
5,tfidf_kmeans_k6,6,-0.4737,0.8167,0.2149,0.9730
10,e5_kmeans_k6,6,-0.0045,0.4167,0.2074,0.2328


Dipilih: nmf_k4


71

## 06 · Audit periode akhir & tren topik

Tidak refit topik pada seluruh rentang setelah evaluasi; ID topik tetap stabil. Persentase termasuk topik -1=unknown. Rolling bersifat trailing, bukan centered smoothing yang melihat masa depan.

In [12]:
selected=models[chosen];data['topic']=selected['labels']
show('Top words train',pd.DataFrame({str(i):vocab[terms] for i,terms in selected['terms'].items()}))
print('NPMI validation:',coherence(selected['terms'],counts_all[middle.to_numpy()]),'| akhir:',coherence(selected['terms'],counts_all[late.to_numpy()]))
shares=pd.crosstab(data.period,data.topic,normalize='index')
enough=data.groupby('period').size().ge(MIN_PERIOD)
show('Proporsi topik mentah (%)',shares*100)
smooth=shares.rolling(3,min_periods=1).mean().where(enough,axis=0)
show('Proporsi trailing 3 periode · bin kurang data=NaN',smooth*100)
from scipy.spatial.distance import jensenshannon
shift=pd.Series({shares.index[i]:jensenshannon(shares.iloc[i-1],shares.iloc[i],base=2) for i in range(1,len(shares))})
show('Perubahan komposisi terbesar · bukan kausalitas',shift.nlargest(TOP))
for topic in sorted(data.topic.unique()):show(f'Contoh topik {topic}',data.loc[data.topic.eq(topic),['time','text']].head(3))
shares.to_csv(OUT/'topic_share.csv');board.to_csv(OUT/'topic_models.csv',index=False)


Top words train


,0,1,2,3
0,tidak,bem,anak,mbg
1,rakyat,ugm,mas,tidak
2,presiden,ketua,rakyat,ada
3,orang,rakyat,terus,rakyat
4,negara,tidak,mahasiswa,maling
5,prabowo,ijazah,allah,korupsi
6,ada,palsu,semoga,program
7,bodoh,ada,muda,nya
8,kalau,jokowi,cerdas,gizi
9,bisa,prabowo,maju,uang


NPMI validation: 0.09794720712128091 | akhir: 0.1480638595824182

Proporsi topik mentah (%)


topic,-1,0,1,2,3
period,,,,,
2026-02-16,2.1145,52.9180,12.8559,21.2010,10.9106
2026-02-23,1.6610,48.5077,9.4991,24.2668,16.0654
2026-03-02,1.1834,41.4201,15.3846,30.1775,11.8343
2026-03-09,4.3796,54.7445,5.1095,24.8175,10.9489
2026-03-16,2.0000,57.3333,10.0000,21.3333,9.3333
2026-03-23,1.7647,51.1765,11.7647,27.6471,7.6471
2026-03-30,1.6529,52.0661,9.0909,21.4876,15.7025
2026-04-06,3.1579,37.8947,8.4211,37.8947,12.6316
2026-04-13,3.1088,38.3420,14.5078,32.6425,11.3990



Proporsi trailing 3 periode · bin kurang data=NaN


topic,-1,0,1,2,3
period,,,,,
2026-02-16,2.1145,52.9180,12.8559,21.2010,10.9106
2026-02-23,1.8878,50.7128,11.1775,22.7339,13.4880
2026-03-02,1.6530,47.6152,12.5799,25.2151,12.9368
2026-03-09,2.4080,48.2241,9.9977,26.4206,12.9495
2026-03-16,2.5210,51.1660,10.1647,25.4428,10.7055
2026-03-23,2.7148,54.4181,8.9581,24.5993,9.3098
2026-03-30,1.8059,53.5253,10.2852,23.4893,10.8943
2026-04-06,2.1918,47.0458,9.7589,29.0098,11.9937
2026-04-13,2.6399,42.7676,10.6732,30.6749,13.2443


... (8 outputs left out)


Contoh topik 2


,time,text
4,2026-02-17 15:57:42+00:00,"MANTAB...tetap semangat,berjuang dengan suara lantang."
11,2026-02-18 03:05:04+00:00,DPRD dan DPR-RI juga termasuk gol yg tak mampu. .....yaitu tak mampu memikirkan rakyat utk maju.
28,2026-02-18 06:07:19+00:00,Mahasiswa segera turun



Contoh topik 3


,time,text
1,2026-02-17 13:40:42+00:00,"Betul skali, MBG ladang korupsi nyata menu sampe ke anak"" tidak sesuai budget. 😡😡😡"
2,2026-02-17 14:51:57+00:00,Pemahaman saya MBG itu tidak gratis Karena dari kami bayar pajak Jadi kami yang membayarnya
6,2026-02-17 21:06:34+00:00,"Jls x MBG program cari modal untuk 2029 n kantong cuan para pejabat , ujungnya suruh byr pajak ini itu ke rakyat Ind..."


## 07 · Tiga estimasi sentimen

Sampel acak di setiap periode agar coverage waktu jelas. Leksikon hanya baseline sederhana. Model classifier dan NLI dapat salah domain/sarkasme; softmax bukan bukti label benar. Tidak mengisi baris yang belum dinilai menjadi netral.

In [13]:
quota=max(1,N_SENTIMENT//data.period.nunique())
sentiment=pd.concat([g.sample(min(quota,len(g)),random_state=SEED) for _,g in data.groupby('period')]).copy()
canonical=['negative','neutral','positive']
POS=set('bagus mantap baik setuju dukung sukses hebat enak'.split());NEG=set('buruk jelek gagal korupsi keracunan bohong rugi parah tolak'.split());NEGATE=set('tidak tak bukan belum jangan tanpa kurang ga gak nggak'.split())
def lexical_sent(text):
    words=re.findall(r'[^\W\d_]+',text.casefold());score=0
    for i,w in enumerate(words):
        s=int(w in POS)-int(w in NEG)
        score+=-s if set(words[max(0,i-3):i])&NEGATE else s
    return 2 if score>0 else 0 if score<0 else 1
pred={'lexicon':np.array([lexical_sent(t) for t in sentiment.text])}
repo='w11wo/indonesian-roberta-base-sentiment-classifier'
tok=AutoTokenizer.from_pretrained(repo);model=AutoModelForSequenceClassification.from_pretrained(repo).cuda().eval()
label_map={k.lower():v for k,v in model.config.label2id.items()};assert set(canonical).issubset(label_map)
prob=[]
with torch.inference_mode():
    for start in range(0,len(sentiment),32):
        enc=tok(sentiment.text.iloc[start:start+32].tolist(),padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):logits=model(**enc).logits
        prob.append(softmax(logits.float().cpu().numpy()[:,[label_map[k] for k in canonical]],axis=1))
prob=np.concatenate(prob);pred['sentiment_pretrained']=prob.argmax(1)
del model;gc.collect();torch.cuda.empty_cache()

In [14]:
hypotheses=['Teks ini menyampaikan keluhan, kritik, atau penolakan.','Teks ini hanya menyampaikan informasi netral.','Teks ini menyampaikan pujian, kepuasan, atau dukungan.']
pairs=pd.DataFrame([{'a':text,'b':hyp} for text in sentiment.text for hyp in hypotheses])
logits,mapping=pair_logits('MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',pairs,'mdeberta_nli')
assert 'entailment' in mapping
nli_prob=softmax(logits[:,mapping['entailment']].reshape(len(sentiment),3),axis=1)
pred['nli_zero_shot']=nli_prob.argmax(1)
for name,value in pred.items():sentiment[name]=value
sentiment['polarity_pretrained']=prob[:,2]-prob[:,0]
sentiment['polarity_nli']=nli_prob[:,2]-nli_prob[:,0]
show('Coverage sentimen · n sumber vs n dinilai',data.groupby('period').size().rename('n_sumber').to_frame().join(sentiment.groupby('period').size().rename('n_dinilai')))
show('Agreement, bukan accuracy',pd.DataFrame({a:{b:np.mean(pred[a]==pred[b]) for b in pred} for a in pred}))


Coverage sentimen · n sumber vs n dinilai


,n_sumber,n_dinilai
period,,
2026-02-16,3547,78
2026-02-23,3853,78
2026-03-02,169,78
2026-03-09,137,78
2026-03-16,150,78
2026-03-23,170,78
2026-03-30,121,78
2026-04-06,95,78
2026-04-13,193,78



Agreement, bukan accuracy


,lexicon,sentiment_pretrained,nli_zero_shot
lexicon,1.0000,0.1539,0.0942
sentiment_pretrained,0.1539,1.0000,0.7539
nli_zero_shot,0.0942,0.7539,1.0000


## 08 · Tren, ketidakpastian sampel & kelemahan

Interval mean adalah sampling SE untuk skor model, bukan interval akurasi sentimen. Komentar duplikat/serial dependence dapat membuatnya terlalu sempit; n_unique disertakan. Periode kecil tetap terlihat dengan flag.

In [15]:
rows=[]
for period,g in sentiment.groupby('period'):
    n=len(g);mean=g.polarity_pretrained.mean();se=g.polarity_pretrained.std()/np.sqrt(n)
    rows.append({'period':period,'n_dinilai':n,'n_unique':g.text.map(key).nunique(),'n_sumber':int((data.period==period).sum()),
        'negatif_%':g.sentiment_pretrained.eq(0).mean()*100,'netral_%':g.sentiment_pretrained.eq(1).mean()*100,
        'positif_%':g.sentiment_pretrained.eq(2).mean()*100,'mean_polarity':mean,'sampling_lo':mean-1.96*se,'sampling_hi':mean+1.96*se,
        'agreement_nli':g.sentiment_pretrained.eq(g.nli_zero_shot).mean(),'cukup_sampel':n>=MIN_PERIOD})
trend=pd.DataFrame(rows).set_index('period').sort_index()
trend['trailing_3']=trend.mean_polarity.rolling(3,min_periods=1).mean()
show('Tren sentimen & dukungan',trend)
show('Perubahan skor terbesar',trend.mean_polarity.diff().abs().nlargest(5))
show('Contoh model tidak sepakat',sentiment.loc[sentiment.sentiment_pretrained.ne(sentiment.nli_zero_shot),['time','text','sentiment_pretrained','nli_zero_shot']].head(TOP))
trend.to_csv(OUT/'sentiment_trend.csv');sentiment.to_csv(OUT/'sentiment_sample.csv',index=False)
print('Tidak mengklaim model paling akurat tanpa gold label; gunakan contoh tidak sepakat untuk audit manusia.')


Tren sentimen & dukungan


,n_dinilai,n_unique,n_sumber,negatif_%,netral_%,positif_%,mean_polarity,sampling_lo,sampling_hi,agreement_nli,cukup_sampel,trailing_3
period,,,,,,,,,,,,
2026-02-16,78,78,3547,70.5128,5.1282,24.3590,-0.4761,-0.6385,-0.3136,0.7821,True,-0.4761
2026-02-23,78,77,3853,84.6154,5.1282,10.2564,-0.7273,-0.8438,-0.6109,0.7949,True,-0.6017
2026-03-02,78,59,169,67.9487,7.6923,24.3590,-0.4609,-0.6184,-0.3034,0.7949,True,-0.5548
2026-03-09,78,70,137,74.3590,11.5385,14.1026,-0.5905,-0.7300,-0.4509,0.7564,True,-0.5929
2026-03-16,78,68,150,75.6410,2.5641,21.7949,-0.5167,-0.6808,-0.3525,0.7821,True,-0.5227
2026-03-23,78,66,170,76.9231,6.4103,16.6667,-0.5685,-0.7234,-0.4136,0.8333,True,-0.5585
2026-03-30,78,60,121,75.6410,0.0000,24.3590,-0.4744,-0.6397,-0.3092,0.8333,True,-0.5199
2026-04-06,78,53,95,64.1026,8.9744,26.9231,-0.3667,-0.5297,-0.2036,0.6154,True,-0.4699
2026-04-13,78,69,193,69.2308,2.5641,28.2051,-0.4188,-0.5953,-0.2422,0.8462,True,-0.4200



Perubahan skor terbesar


,mean_polarity
period,
2026-03-02,0.2665
2026-02-23,0.2513
2026-06-22,0.1984
2026-05-04,0.1556
2026-03-09,0.1296



Contoh model tidak sepakat


,time,text,sentiment_pretrained,nli_zero_shot
2131,2026-02-20 05:49:28+00:00,MBG = MAKAN BASI GRATIS = MAKAN BERACUN GRATIS .,1,2
3080,2026-02-22 09:10:19+00:00,"Anak muda cerdas calon pemimpin Indonesia...apa yang dipaparkan semua benar adanya, berkata jujur demi keadilan untu...",2,0
781,2026-02-19 02:13:52+00:00,TOP MARKOTOP.... POKOKNYA TOP DEH.... TEROR SUDAH PASTI SEJARAH SUDAH MEMBUKTIKAN AKTIFIS DI NEGARA INI BANYAK YANG ...,2,0
1739,2026-02-19 15:21:33+00:00,Ini baru Mahasiswa,1,0
3188,2026-02-22 11:49:45+00:00,"Dari dulu begitu usai kuliah masuk partai ,duduk di DPR diam",2,1
443,2026-02-18 22:22:26+00:00,setuju g kerja hanya janji mulu,0,1
3298,2026-02-22 13:50:24+00:00,Ini baru maha sewa Anis bas edan tau semua 👎👎👎👎👎👎👎👎👎👎,2,0
70,2026-02-18 11:17:09+00:00,Sebaiknya buat angket saja dan yg evaluasi bukan dari partai GERINDRA tapi dari partai oposisi,1,0
1200,2026-02-19 06:59:15+00:00,Maju terus mas ket BEM..,0,2
1075,2026-02-19 05:25:08+00:00,Jokowi prabowo saling melindungi kejahatan nya....dua2nya pembawa bencana,1,0


Tidak mengklaim model paling akurat tanpa gold label; gunakan contoh tidak sepakat untuk audit manusia.


## Interpretasi & sumber
Semua angka dihitung saat run; pemilihan validation tidak diubah setelah holdout. E5 memakai query: di kedua sisi STS, query:/passage: pada retrieval. [E5](https://huggingface.co/intfloat/multilingual-e5-base), [IndoBERT](https://huggingface.co/indobenchmark/indobert-base-p1), [IndoNLI](https://github.com/ir-nlp-csui/indonli).

In [16]:
import importlib.metadata as metadata
manifest={'task':RESULT_NAME,'seconds':time.perf_counter()-START,'gpu':torch.cuda.get_device_name(0),'seed':SEED,'versions':{p:metadata.version(p) for p in ['torch','transformers','scikit-learn','pandas']}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(manifest)

{'task': 'hasil_temporal_review', 'seconds': 101.68241387700004, 'gpu': 'Tesla T4', 'seed': 42, 'versions': {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'scikit-learn': '1.6.1', 'pandas': '2.3.3'}}
